# Compound SV / SHAP mappping

In [ ]:
%load_ext autoreload

In [5]:
import sys
from pathlib import Path

# Resolve workspace root (works whether cwd is notebooks/ or the repo root)
_ws_root = Path.cwd()
if not (_ws_root / "src").exists():
    _ws_root = _ws_root.parent

_servers_dir = _ws_root / "src" / "chemagent" / "servers"

for _p in [str(_ws_root), str(_servers_dir)]:
    if _p not in sys.path:
        sys.path.insert(0, _p)

_save_dir = _ws_root / "notebooks" / "debug_outputs"
_save_dir.mkdir(exist_ok=True)

print(f"Workspace root : {_ws_root}")
print(f"Servers dir    : {_servers_dir}")
print(f"Save dir       : {_save_dir}")

Workspace root : c:\Users\janela\OneDrive - uni-bonn.de\Code\AI-Agent-for-Compound-Selectivity-Prediction-and-Explainability
Servers dir    : c:\Users\janela\OneDrive - uni-bonn.de\Code\AI-Agent-for-Compound-Selectivity-Prediction-and-Explainability\src\chemagent\servers
Save dir       : c:\Users\janela\OneDrive - uni-bonn.de\Code\AI-Agent-for-Compound-Selectivity-Prediction-and-Explainability\notebooks\debug_outputs


# Import libraries

In [6]:
import pandas as pd
from rdkit import Chem
from src.chemagent.explainability.mol_shap_draw import shap_to_atom_weight, get_ecfp_morgan_generator_bit_info, get_atom_wise_weight_map


# Load data

In [ ]:
df_shap = pd.read_pickle('df_shap_sample.pkl')
df_shap

# Generate MOL, bit info and atom wise shap values

In [ ]:
smiles = df_shap["smiles"].iloc[0]
smiles

In [ ]:
mol = Chem.MolFromSmiles(smiles)
mol 

# Genererate Mol bit info

In [ ]:
n_bits = 2048

bit_info = get_ecfp_morgan_generator_bit_info(smiles, n_bits=n_bits, radius=2)
bit_info

# Get Shap values

In [ ]:
shap_values = df_shap.iloc[0]['shap_values']
shap_values

# Shap to atom weight

In [ ]:
shap_value_to_atom_weight = shap_to_atom_weight(mol=mol, dict_bit_info=bit_info, shapley_values=shap_values)
shap_value_to_atom_weight

# Shap mapping

In [ ]:
shap_mapping = get_atom_wise_weight_map(mol=mol, weights=shap_value_to_atom_weight, mol_size=(1000, 800))
shap_mapping

In [ ]:
shap_mapping.save('shap_mapping.png')

In [7]:
from src.chemagent.servers.chemagent_mcp import explain_with_shap, plot_shap_mol, explain_smiles_with_shap
import os
import numpy as np

In [8]:
import joblib
model_path = os.path.join(_ws_root, 'data/logs/session_tiago_20260312_104021_8f6812/models/chembl_activity_data_O00329_P48736_scaffold_RFC.pkl')
model_path

'c:\\Users\\janela\\OneDrive - uni-bonn.de\\Code\\AI-Agent-for-Compound-Selectivity-Prediction-and-Explainability\\data/logs/session_tiago_20260312_104021_8f6812/models/chembl_activity_data_O00329_P48736_scaffold_RFC.pkl'

In [9]:
split_file_path = os.path.join(_ws_root, './data/logs/session_tiago_20260312_104021_8f6812/splits/chembl_activity_data_O00329_P48736_scaffold.pkl')

In [10]:
model = joblib.load(model_path)
split_data = joblib.load(split_file_path)
split_data

{'train_features': array([[0, 0, 0, ..., 0, 0, 0],
        [0, 1, 1, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        ...,
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0]], shape=(1021, 2048)),
 'train_labels': array([0, 0, 0, ..., 0, 0, 1], shape=(1021,)),
 'val_features': array([], shape=(0, 2048), dtype=int64),
 'val_labels': array([], dtype=int64),
 'test_features': array([[0, 1, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        ...,
        [0, 0, 0, ..., 0, 0, 0],
        [0, 1, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0]], shape=(256, 2048)),
 'test_labels': array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 2, 0, 0, 0, 1,
        1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0,

In [11]:
split_data['test_smiles'][10:11]

<StringArray>
['Cc1cc(-c2ccn3nc(N)c(C(=O)NC4COC4)c3n2)cc2c1C(=O)N(C(C)C1CC1)C2']
Length: 1, dtype: str

In [12]:
split_data['test_labels'][10:11]

array([2])

In [13]:
split_data_test = np.array(split_data[f"test_features"])[:1]
X_train     = np.array(split_data["train_features"])[:1]
split_data_test

array([[0, 1, 0, ..., 0, 0, 0]], shape=(1, 2048))

In [14]:
prediction = model.predict(split_data_test)
prediction

array([0])

In [15]:
# Use SHAP to explain the model
import shap
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(split_data_test)
shap_values_predicted_class = shap_values
shap_values_predicted_class

array([[[ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
        [-7.77078571e-03,  6.86607455e-03,  9.04711162e-04],
        [-9.65621664e-06,  9.65621664e-06,  0.00000000e+00],
        ...,
        [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
        [-1.51333820e-06,  1.51333820e-06,  0.00000000e+00],
        [-1.62122217e-05,  0.00000000e+00,  1.62122217e-05]]],
      shape=(1, 2048, 3))

In [16]:
from src.chemagent.explainability.shap_explainer import SHAPExplainer
explainer    = SHAPExplainer.from_model_path(model_path, background=X_train)
shap_values  = explainer.explain(split_data_test)  # (n_samples, n_features)
expected_val = explainer.expected_value

print("SHAP values:", shap_values)
print("Expected value:", expected_val)

SHAP values: [[[ 0.00000000e+00  0.00000000e+00  0.00000000e+00]
  [-7.77078571e-03  6.86607455e-03  9.04711162e-04]
  [-9.65621664e-06  9.65621664e-06  0.00000000e+00]
  ...
  [ 0.00000000e+00  0.00000000e+00  0.00000000e+00]
  [-1.51333820e-06  1.51333820e-06  0.00000000e+00]
  [-1.62122217e-05  0.00000000e+00  1.62122217e-05]]]
Expected value: 0.742722820763957


In [17]:
shap_values[:, :, 0]

array([[ 0.00000000e+00, -7.77078571e-03, -9.65621664e-06, ...,
         0.00000000e+00, -1.51333820e-06, -1.62122217e-05]],
      shape=(1, 2048))

In [18]:
shap_values_perclass  = explainer.explain_per_predicted_class(split_data_test, prediction)
shap_values_perclass

array([[ 0.00000000e+00, -7.77078571e-03, -9.65621664e-06, ...,
         0.00000000e+00, -1.51333820e-06, -1.62122217e-05]],
      shape=(1, 2048))

In [19]:
mean_abs_per_feature = np.abs(shap_values_perclass).mean(axis=0)
top_bits = np.argsort(mean_abs_per_feature)[::-1][:10].tolist()
top_bits

[358, 1136, 1384, 1620, 790, 708, 674, 213, 690, 1810]

In [ ]:
explain_with_shap(model_path=model_path, 
                  split_file_path=split_file_path, 
                  split='test',
                  save_path=os.path.join(_save_dir, 'shap_mapping_test.pkl')
                  )

In [21]:
explain_smiles_with_shap(model_path=model_path,
                        smiles=split_data['test_smiles'][10:11],
                        )

{'shap_values_path': 'C:\\Users\\janela\\OneDrive - uni-bonn.de\\Code\\AI-Agent-for-Compound-Selectivity-Prediction-and-Explainability\\data\\logs\\session_tiago_20260320_141212_ea4d71\\results\\chembl_activity_data_O00329_P48736_scaffold_RFC_smiles_shap.pkl',
 'n_samples': 1,
 'n_features': 2048,
 'expected_value': 0.742722820763957,
 'prediction': 2,
 'mean_abs_shap': 0.0004890637264829797,
 'shap_sum': 0.9557296767876031,
 'method': 'ECFP',
 'has_smiles': True,
 'note': 'Labels in output file are model predictions, not ground truth.',
 'next_step': "Call plot_shap_mol('C:\\Users\\janela\\OneDrive - uni-bonn.de\\Code\\AI-Agent-for-Compound-Selectivity-Prediction-and-Explainability\\data\\logs\\session_tiago_20260320_141212_ea4d71\\results\\chembl_activity_data_O00329_P48736_scaffold_RFC_smiles_shap.pkl') to render atom-level SHAP heatmaps. Labels shown reflect the model's predicted class."}

In [ ]:
plot_shap_mol(shap_values_path=os.path.join(_save_dir, 'shap_mapping_test.pkl'))

In [ ]:
_save_dir